# Fine-Tune DeepSeek-R1-Distill-Qwen-1.5B on Incident Response Dataset
This notebook fine-tunes the `DeepSeek-R1-Distill-Qwen-1.5B` model for binary classification (`malicious` vs `benign`) using your `fine_tuning_dataset.jsonl`. Training utilizes Apple Silicon or Colab runtime as available.


In [ ]:
# Install dependencies (run once)
!pip install --upgrade pip
!pip install transformers datasets scikit-learn torch torchvision accelerate

In [ ]:
# 1️⃣ Imports & Config
import os
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Configuration
DATASET_JSONL = "fine_tuning_dataset.jsonl"
MODEL_NAME     = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
OUTPUT_DIR     = "./deepseek_qwen_ft"
NUM_LABELS     = 2

# Device
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 2️⃣ Load & Preprocess the Dataset
import pandas as pd
# Read JSONL via pandas
df = pd.read_json(DATASET_JSONL, lines=True)
# Convert to a 🤗 Dataset
ds = Dataset.from_pandas(df)
# Split into train/test
dataset = ds.train_test_split(test_size=0.1, seed=42)

# Map text labels to integers
def map_labels(example):
    lab = example['output'].strip().lower()
    return {'label': 1 if lab == 'malicious' else 0}
dataset = dataset.map(map_labels)

# ─── Tokenize & prepare for Trainer ───
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True
)
def tokenize_fn(examples):
    return tokenizer(
        examples['input'], truncation=True, padding='longest'
    )
dataset = dataset.map(tokenize_fn, batched=True)

# Rename label -> labels and format for PyTorch
dataset = dataset.rename_column('label', 'labels')
dataset.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

# Show summary
print(dataset)
print("\nExample entry:", dataset['train'][0])

In [ ]:
# 3️⃣ Define Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

In [ ]:
# 4️⃣ Load Model & Setup Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, trust_remote_code=True, num_labels=NUM_LABELS
)
model.to(device)
# Ensure pad token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=200,
    save_steps=200,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    bf16=False,
    dataloader_pin_memory=False,
    num_train_epochs=3,
    logging_dir='./logs_deepseek',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
# 5️⃣ Train and Save
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model and tokenizer saved to {OUTPUT_DIR}")